In [ ]:
import os
if os.path.basename(os.getcwd()) == "diagnostic_files":
    os.chdir("..")                       # run from the project root

In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import numpy as np
from generator import make_scenario, simulate_disclosure
from env import play, score
from rational import (price_grid, prior_G, client_policy, dealer_expected_profit,
                      dealer_best_quote, rational_bots, predictions)
from benchmarks import client_belief

CAL = yaml.safe_load(open("config.yaml"))["calibration"]
G, D = CAL["g_points"], CAL["disclosure"]
SEEDS = range(1, 1001)                      # never 1001+ (held out)
SCN = [make_scenario(sd, G) for sd in SEEDS]
SURPLUS = np.array([s.r - s.c for s in SCN])
print(f"g = {G}, {len(SCN)} scenarios")


In [ ]:
# VoD at the locked band settings

def run(tau):
    """U*, B*, VoD for every scenario, with the band built at this tau."""
    out = []
    for scn in SCN:
        band = simulate_disclosure(scn, G, D["K"], D["omega"], tau)
        G_band = client_belief(price_grid(scn.m, G), scn, G, band)
        out.append(predictions(scn, G, G_band))
    return out

def summarise(v, surplus):
    same = np.abs(v) < 1e-12
    return (f"VoD = {v.sum() / surplus.sum():+.1%} of surplus | "
            f"helped {np.sum((v > 0) & ~same)}, hurt {np.sum((v < 0) & ~same)}, same {same.sum()}")

main = run(D["tau"])
U = [u for u, _, _ in main]
B = [b for _, b, _ in main]
vod = np.array([v for *_, v in main])

print("seeds 1-1000:", summarise(vod, SURPLUS))
print("seeds 1-500: ", summarise(vod[:500], SURPLUS[:500]))
for name, rows in (("U*", U), ("B*", B)):
    cap = np.mean([o.client_capture for o in rows])
    lifted = np.mean([o.status == "quote_accepted" for o in rows])
    nd = np.mean([o.no_deal for o in rows])
    print(f"{name}: mean capture {cap:.3f} | quote lifted {lifted:.0%} | no deal {nd:.1%}")


### Interpretation of VoD with the band settings

--- Seeds 1-1000 [Reference A] --- 
A perfect client holding the band takes, on average, an extra 11.7% of the available surplus (client_limit - dealer_cost), averaging 2.24g. 

In 576 scenarios, the band improved the client's result. In 77 cases, it made it worse. In 347 cases, it changed nothing. The band therefore improves the average, but does not guarantee a win in every trade.

--- Seeds 1-500 [Reference B] ---
Results support the above findings, where the perfect client holding the band has, on average, a 12% surplus. The band helped in 279 of cases.

### Interpretation of U* and B*

U* and B* represents the environments played with and without the band (U and B respectively). Both environments assume a rational dealer that quotes at p* and accepts any counter at or above its cost.

In environment U*, the client's belief about the dealer's cost is the prior only ("typically 0.53g–1.9g below m"). The client plays optimally given its belief - lifts the quote if it's at or below its walkaway point theta, otherwise counters at q*.

In B*, we introduce the Bayesian posterior (prior with the band) to the client's belief about the dealer's cost. The client also plays optimally here.

There value of disclosure (VoD), the most the band can be worth to a client that uses it perfectly, is therefore B* - U*. In Reference A, the VoD is 11.7%. 

This VoD will be compared against the LLM's B-U as a percentage, where we introduce E_client (how much the LLM captured as a percentage of the total possible surplus capture).


In [ ]:
#Does the quote reveal the cost? This is what a perfect inverse planner could learn from the quote

p_g = np.array([(dealer_best_quote(s.m, s.c, G) - s.m) / G for s in SCN])
c_g = np.array([(s.m - s.c) / G for s in SCN])          # cost, in g below m

key = np.round(p_g, 6)
vals, counts = np.unique(key, return_counts=True)
print(f"p* takes {len(vals)} distinct values; most common m{vals[counts.argmax()]:+.3f}g "
      f"in {counts.max() / len(p_g):.0%} of scenarios")
print(f"p* range m{p_g.min():+.3f}g to m{p_g.max():+.3f}g | corr(p*, c) = {np.corrcoef(p_g, -c_g)[0, 1]:+.2f}")

# best possible reading of the quote: predict c by the average c at that quote
fitted = np.array([c_g[key == k].mean() for k in key])
print(f"R^2 of c on p* = {1 - np.var(c_g - fitted) / np.var(c_g):.2f}")

for lo, hi in [(0, 0.53), (0.53, 1), (1, 1.9), (1.9, 9)]:
    sel = (c_g >= lo) & (c_g < hi)
    print(f"  cost m-{lo}g .. m-{hi}g  n={sel.sum():4d}  mean p* = m{p_g[sel].mean():+.3f}g")


#### Interpretation of what the quote reveals about the cost

Evidence supports that the quote barely depends on the cost. 

Across 1,000 dealers with costs anywhere from m to 3g or more below it, 880 dealers quote exactly m - 0.425g. Since every dealer shows the same price, the price can't tell you which dealer you're facing. 

Regarding the correlation between p* and c: the generator is created so that the middle 80% of costs spans about 1.4g (0.51g to 1.95g), and the full range is about 4.4g. However, the quote spans only 0.275g. The correlation between cost and pricing is also weak, despite being in the right direction (a higher cost has a higher quote).

Most importantly, a R^2 of 0.18 tells us that the quote removes only 18% of the uncertainty about the dealer's cost. 

So, this proves that against the rational dealer, the quote carries little information. So if the LLM client's belief about cost moves a lot after seeing the quote, it's reading more into the quote than the benchmark says is there.

In [ ]:
# How sensitive VoD is to tau

print(f"{'tau':>5} {'s/g':>6}  result")
for tau in (0.0, 0.1, 0.25, 0.4, 0.6):
    v = vod if tau == D["tau"] else np.array([x for *_, x in run(tau)])
    s_g = np.sqrt(D["omega"] ** 2 / D["K"] + tau ** 2)
    print(f"{tau:>5} {s_g:>6.3f}  {summarise(v, SURPLUS)}")


In [ ]:
# Ceiling for VoI (what the dealer would gain if it knew the client's exact limit)

def quote_knowing_r(r):
    """Dealer's best quote if it knew this client's limit: F0 replaced by a point mass at r."""
    def quote(dv, g):
        q = price_grid(dv.m, g)
        theta, q_star = client_policy(np.array([r]), q, prior_G(q, dv.m, g))
        ev = dealer_expected_profit(dv.c, q, np.array([1.0]), theta, q_star)
        return float(q[np.argmax(np.where(q >= dv.c, ev, -np.inf))])
    return quote

gain = []
for scn, u in zip(SCN, U):
    q = price_grid(scn.m, G)
    _, respond, decide = rational_bots(q, prior_G(q, scn.m, G))     # client in U
    k = score(scn, play(scn, G, quote_knowing_r(scn.r), respond, decide))
    gain.append(k.dealer_surplus - u.dealer_surplus)
gain = np.array(gain)
print(f"VoI ceiling (dealer knows r, condition U): {gain.sum() / SURPLUS.sum():+.1%} of surplus")